In [1]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [2]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cpu device


In [3]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [4]:
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [5]:
X = torch.rand(1, 28, 28, device=device)
logits = model(X)
pred_probab = nn.Softmax(dim=1)(logits)
y_pred = pred_probab.argmax(1)
print(f"Predicted class: {y_pred}")

Predicted class: tensor([9])


In [6]:
input_image = torch.rand(3,28,28)
print(input_image.size())

torch.Size([3, 28, 28])


In [7]:
flatten = nn.Flatten()
flat_image = flatten(input_image)
print(flat_image.size())

torch.Size([3, 784])


In [8]:
layer1 = nn.Linear(in_features=28*28, out_features=20)
hidden1 = layer1(flat_image)
print(hidden1.size())

torch.Size([3, 20])


In [9]:
print(f"Before ReLU: {hidden1}\n\n")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")

Before ReLU: tensor([[-0.5996,  0.1133,  0.3065,  0.2143,  0.1676,  0.5682, -0.7460, -0.0866,
         -0.0680,  0.0213, -0.3140,  0.4133,  0.1475,  0.2843, -0.3788, -0.4366,
          0.1309,  0.2144, -0.2072, -0.2225],
        [-0.4310,  0.1686,  0.4466,  0.1041, -0.0551,  0.5421, -0.2507, -0.1320,
          0.0452,  0.0251, -0.4245,  0.0822,  0.2760, -0.1030, -0.2716, -0.2264,
          0.2627,  0.2356, -0.4564, -0.3091],
        [-0.6960, -0.2852,  0.2929,  0.0141,  0.1371,  0.4703, -0.6135, -0.0379,
          0.1457,  0.0578, -0.3212,  0.1673,  0.1158, -0.2720, -0.3481, -0.2617,
         -0.1862, -0.0083, -0.3459, -0.1331]], grad_fn=<AddmmBackward0>)


After ReLU: tensor([[0.0000, 0.1133, 0.3065, 0.2143, 0.1676, 0.5682, 0.0000, 0.0000, 0.0000,
         0.0213, 0.0000, 0.4133, 0.1475, 0.2843, 0.0000, 0.0000, 0.1309, 0.2144,
         0.0000, 0.0000],
        [0.0000, 0.1686, 0.4466, 0.1041, 0.0000, 0.5421, 0.0000, 0.0000, 0.0452,
         0.0251, 0.0000, 0.0822, 0.2760, 0.0000, 0.00

In [10]:
seq_modules = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20, 10)
)
input_image = torch.rand(3,28,28)
logits = seq_modules(input_image)

In [11]:
softmax = nn.Softmax(dim=1)
pred_probab = softmax(logits)

In [12]:
print(f"Model structure: {model}\n\n")

for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n")

Model structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Values : tensor([[-0.0135, -0.0176, -0.0223,  ...,  0.0353, -0.0339,  0.0288],
        [-0.0172, -0.0197, -0.0344,  ..., -0.0157, -0.0231,  0.0039]],
       grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) | Values : tensor([-0.0131,  0.0126], grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Values : tensor([[ 0.0429, -0.0022,  0.0063,  ..., -0.0406, -0.0058,  0.0442],
        [-0.0357,  0.0213,  0.0336,  ..., -0.0120, -0.0079,  0.0347]],
       grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.bias | 

In [13]:
import torch

x = torch.ones(5)  # input tensor
y = torch.zeros(3)  # expected output
w = torch.randn(5, 3, requires_grad=True)
b = torch.randn(3, requires_grad=True)
z = torch.matmul(x, w)+b
loss = torch.nn.functional.binary_cross_entropy_with_logits(z, y)

In [14]:
print(f"Gradient function for z = {z.grad_fn}")
print(f"Gradient function for loss = {loss.grad_fn}")

Gradient function for z = <AddBackward0 object at 0x72e9fa49e800>
Gradient function for loss = <BinaryCrossEntropyWithLogitsBackward0 object at 0x72e9fa682050>


In [15]:
loss.backward()
print(w.grad)
print(b.grad)

tensor([[0.1435, 0.0084, 0.0041],
        [0.1435, 0.0084, 0.0041],
        [0.1435, 0.0084, 0.0041],
        [0.1435, 0.0084, 0.0041],
        [0.1435, 0.0084, 0.0041]])
tensor([0.1435, 0.0084, 0.0041])


In [16]:
z = torch.matmul(x, w)+b
print(z.requires_grad)

with torch.no_grad():
    z = torch.matmul(x, w)+b
print(z.requires_grad)

True
False


In [17]:
z = torch.matmul(x, w)+b
z_det = z.detach()
print(z_det.requires_grad)

False


In [18]:
inp = torch.eye(4, 5, requires_grad=True)
out = (inp+1).pow(2).t()
out.backward(torch.ones_like(out), retain_graph=True)
print(f"First call\n{inp.grad}")
out.backward(torch.ones_like(out), retain_graph=True)
print(f"\nSecond call\n{inp.grad}")
inp.grad.zero_()
out.backward(torch.ones_like(out), retain_graph=True)
print(f"\nCall after zeroing gradients\n{inp.grad}")

First call
tensor([[4., 2., 2., 2., 2.],
        [2., 4., 2., 2., 2.],
        [2., 2., 4., 2., 2.],
        [2., 2., 2., 4., 2.]])

Second call
tensor([[8., 4., 4., 4., 4.],
        [4., 8., 4., 4., 4.],
        [4., 4., 8., 4., 4.],
        [4., 4., 4., 8., 4.]])

Call after zeroing gradients
tensor([[4., 2., 2., 2., 2.],
        [2., 4., 2., 2., 2.],
        [2., 2., 4., 2., 2.],
        [2., 2., 2., 4., 2.]])


In [27]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

train_dataloader = DataLoader(training_data, batch_size=64)
test_dataloader = DataLoader(test_data, batch_size=64)

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork()

In [28]:
learning_rate = 1e-3
batch_size = 64
epochs = 5

In [29]:
# Initialize the loss function
loss_fn = nn.CrossEntropyLoss()

In [30]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    # Set the model to training mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        # Compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


def test_loop(dataloader, model, loss_fn):
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [31]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

epochs = 5
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.305914  [   64/60000]
loss: 2.293356  [ 6464/60000]
loss: 2.275717  [12864/60000]
loss: 2.272499  [19264/60000]
loss: 2.250467  [25664/60000]
loss: 2.236554  [32064/60000]
loss: 2.236097  [38464/60000]
loss: 2.208750  [44864/60000]
loss: 2.201352  [51264/60000]
loss: 2.176562  [57664/60000]
Test Error: 
 Accuracy: 49.5%, Avg loss: 2.170905 

Epoch 2
-------------------------------
loss: 2.173282  [   64/60000]
loss: 2.168705  [ 6464/60000]
loss: 2.116176  [12864/60000]
loss: 2.136602  [19264/60000]
loss: 2.079844  [25664/60000]
loss: 2.037796  [32064/60000]
loss: 2.058201  [38464/60000]
loss: 1.986218  [44864/60000]
loss: 1.980990  [51264/60000]
loss: 1.924552  [57664/60000]
Test Error: 
 Accuracy: 59.1%, Avg loss: 1.919133 

Epoch 3
-------------------------------
loss: 1.943394  [   64/60000]
loss: 1.921075  [ 6464/60000]
loss: 1.808807  [12864/60000]
loss: 1.851266  [19264/60000]
loss: 1.733375  [25664/60000]
loss: 1.692935  [32064/600